# 03｜逐年分析与 2026 表现解释

本 Notebook 只读取 02 的 O2O 加算逐日结果和持有段明细，不重新计算信号。

输出包括：训练/验证/测试期逐年收益、状态分布、持有段收益与胜率、年度信号计数，以及专门解释 2026 表现的逐月拆解、状态覆盖和慢快线/规则轴诊断。

In [1]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display

PACKAGE_ROOT = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / 'src' / 'generate_compact_output.py').is_file()
)
SRC_ROOT = PACKAGE_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

SPOT_TEXT = os.environ.get(
    'COMPANY_SPOT_PATH',
    '/home/hzy/cta/IC数据更新_最终固化版/现货最终版/CSI500_SPOT_md_eod_raw_最终版.parquet',
).strip()
if not SPOT_TEXT or not Path(SPOT_TEXT).expanduser().is_absolute():
    raise RuntimeError('请设置 COMPANY_SPOT_PATH 为本地米筐现货的绝对路径。')
SPOT_PATH = Path(SPOT_TEXT).expanduser().resolve()
STAGE04_DIR = Path(os.environ.get(
    'ANALYSIS_04_OUTPUT_DIR',
    str(PACKAGE_ROOT / 'runtime_outputs_04_returns'),
)).expanduser().resolve()
OUTPUT_DIR = Path(os.environ.get(
    'ANALYSIS_05_OUTPUT_DIR',
    str(PACKAGE_ROOT / 'runtime_outputs_05_yearly'),
)).expanduser().resolve()
PANEL_TEXT = os.environ.get('REMOTE_ANALYSIS_PANEL_PATH', '').strip()
PANEL_PATH = Path(PANEL_TEXT).expanduser().resolve() if PANEL_TEXT else None
if not STAGE04_DIR.is_absolute() or not OUTPUT_DIR.is_absolute():
    raise RuntimeError('03 的 02 输入目录和输出目录都必须是绝对路径。')

from reproduce_remote_o2o import run_stage_05

print('本地现货：', SPOT_PATH)
print('02 输入：', STAGE04_DIR)
print('03 输出：', OUTPUT_DIR)
print('可选 1545 内部面板：', PANEL_PATH if PANEL_PATH else '未显式指定，将从冻结包现货重建')

本地现货： /Users/hzy/Desktop/0817合并查看/99_中间归档/本地数据快照_不入库/_rq_latest_20260824_1750/CSI500_SPOT_md_eod_raw_RQ_20260817.parquet
04 输入： /Users/hzy/Desktop/0817合并查看/20260824_1805_冻结中证500输出包/runtime_outputs_04_returns
05 输出： /Users/hzy/Desktop/0817合并查看/20260824_1805_冻结中证500输出包/runtime_outputs_05_yearly
可选 1545 内部面板： 未显式指定，将从冻结包现货重建


## 1. 读取 04 并生成逐年分析

In [2]:
metadata = run_stage_05(SPOT_PATH, STAGE04_DIR, OUTPUT_DIR, PANEL_PATH)
annual_adj = pd.read_csv(OUTPUT_DIR / '逐年_加入四个反转分析.csv', encoding='utf-8-sig')
annual_raw = pd.read_csv(OUTPUT_DIR / '逐年_原始三状态分析.csv', encoding='utf-8-sig')
display(annual_raw)
display(annual_adj)
print('2026 摘要：', metadata['year_2026'])

/Users/hzy/Desktop/0817合并查看/20260824_1805_冻结中证500输出包/src/reproduce_remote_o2o.py:753: UserWarning: Glyph 21407 (\N{CJK UNIFIED IDEOGRAPH-539F}) missing from current font.
  fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
/Users/hzy/Desktop/0817合并查看/20260824_1805_冻结中证500输出包/src/reproduce_remote_o2o.py:753: UserWarning: Glyph 22987 (\N{CJK UNIFIED IDEOGRAPH-59CB}) missing from current font.
  fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
/Users/hzy/Desktop/0817合并查看/20260824_1805_冻结中证500输出包/src/reproduce_remote_o2o.py:753: UserWarning: Glyph 19977 (\N{CJK UNIFIED IDEOGRAPH-4E09}) missing from current font.
  fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
/Users/hzy/Desktop/0817合并查看/20260824_1805_冻结中证500输出包/src/reproduce_remote_o2o.py:753: UserWarning: Glyph 29366 (\N{CJK UNIFIED IDEOGRAPH-72B6}) missing from current font.
  fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=fig.get_facec

,series,year,phase,strategy_return_pct,index_return_pct,excess_return_pct,short_days,flat_days,long_days,directional_days,winning_directional_days,directional_win_rate_pct,mean_directional_return_pct,days,zero_down_days,zero_up_days,plus_exit_days,minus_exit_days,big_up_days,big_down_days
0,Raw 1545,2018,Train,-5.994147,-38.745579,32.751432,17,208,17,34,16,47.058824,-0.176298,242,40,44,1,1,12,40
1,Raw 1545,2019,Train,44.066750,26.477225,17.589525,11,171,62,73,49,67.123288,0.603654,244,35,27,0,4,17,14
2,Raw 1545,2020,Train,9.357476,22.590673,-13.233197,20,179,44,64,32,50.000000,0.146211,243,42,47,7,6,18,26
3,Raw 1545,2021,Train,6.589651,15.897309,-9.307658,27,175,41,68,40,58.823529,0.096907,243,61,65,5,2,10,2
4,Raw 1545,2022,Train,21.729503,-20.990005,42.719508,59,143,40,99,51,51.515152,0.219490,242,37,53,6,13,10,9
5,Raw 1545,2023,Val,6.642019,-6.531712,13.173731,35,159,48,83,45,54.216867,0.080024,242,27,29,0,7,8,0
6,Raw 1545,2024,Val,28.581021,10.594100,17.986921,28,175,39,67,35,52.238806,0.426582,242,27,63,10,4,13,26
7,Raw 1545,2025,Test,9.905724,29.607073,-19.701349,6,164,73,79,42,53.164557,0.125389,243,32,41,16,1,12,12
8,Raw 1545,2026,Test,22.883916,6.603140,16.280776,30,82,42,72,43,59.722222,0.317832,154,23,21,17,7,13,16


,series,year,phase,strategy_return_pct,index_return_pct,excess_return_pct,short_days,flat_days,long_days,directional_days,winning_directional_days,directional_win_rate_pct,mean_directional_return_pct,days,zero_down_days,zero_up_days,plus_exit_days,minus_exit_days,big_up_days,big_down_days
0,Adj 1545,2018,Train,6.498453,-38.745579,45.244032,48,142,52,100,49,49.000000,0.064985,242,40,44,1,1,12,40
1,Adj 1545,2019,Train,71.376109,26.477225,44.898884,39,119,86,125,84,67.200000,0.571009,244,35,27,0,4,17,14
2,Adj 1545,2020,Train,24.654333,22.590673,2.063660,42,131,70,112,57,50.892857,0.220128,243,42,47,7,6,18,26
3,Adj 1545,2021,Train,9.827413,15.897309,-6.069896,69,90,84,153,90,58.823529,0.064231,243,61,65,5,2,10,2
4,Adj 1545,2022,Train,28.805314,-20.990005,49.795319,68,102,72,140,74,52.857143,0.205752,242,37,53,6,13,10,9
5,Adj 1545,2023,Val,4.482964,-6.531712,11.014677,50,120,72,122,64,52.459016,0.036746,242,27,29,0,7,8,0
6,Adj 1545,2024,Val,59.099990,10.594100,48.505889,46,109,87,133,69,51.879699,0.444361,242,27,63,10,4,13,26
7,Adj 1545,2025,Test,42.934089,29.607073,13.327016,30,122,91,121,72,59.504132,0.354827,243,32,41,16,1,12,12
8,Adj 1545,2026,Test,35.478338,6.603140,28.875198,40,72,42,82,49,59.756098,0.432663,154,23,21,17,7,13,16


2026 摘要： {'rows_2026': 154, 'months_2026': 8, 'total_adjusted_return_pct': 35.47833834278934, 'total_index_return_pct': 6.603140189083378, 'adjusted_active_days': 82}


## 2. 为什么 2026 表现好

这里不把“表现好”归因于单一指标，而是同时查看：O2O 日收益、相对指数超额、方向覆盖天数、每月贡献和冻结引擎的慢快线/规则轴/四维状态。

In [3]:
monthly_2026 = pd.read_csv(OUTPUT_DIR / '2026_逐月表现分解.csv', encoding='utf-8-sig')
detail_2026 = pd.read_csv(OUTPUT_DIR / '2026_逐日表现分解.csv', encoding='utf-8-sig', parse_dates=['实际执行日'])
display(monthly_2026)
display(detail_2026.tail(15))
print('2026 调整后加算收益：', monthly_2026['adjusted_return_pct'].sum(), '%')
print('2026 调整后相对指数超额：', monthly_2026['adjusted_excess_pct'].sum(), '%')
print('2026 调整后方向持有日：', int(detail_2026['调整后三状态'].ne(0).sum()))

,month,days,raw_return_pct,adjusted_return_pct,index_return_pct,raw_active_days,adjusted_active_days,adjusted_winning_days,adjusted_excess_pct
0,2026-01,20,13.170461,9.831906,9.811824,17,14,10,0.020082
1,2026-02,14,0.000000,1.736155,3.741086,0,3,2,-2.004931
2,2026-03,22,4.015302,7.451318,-9.650649,9,10,6,17.101967
3,2026-04,21,5.275633,3.654618,8.533480,10,12,7,-4.878862
4,2026-05,18,1.133128,-0.982001,-0.705635,9,11,5,-0.276366
5,2026-06,21,-0.088736,3.840911,8.283574,9,16,10,-4.442663
6,2026-07,23,-0.488286,10.632596,-18.836583,17,10,6,29.469179
7,2026-08,15,-0.133586,-0.687166,5.426043,1,6,3,-6.113209


,实际执行日,三状态,+1反转,-1反转,0转-1,0转+1,大涨,大跌,调整后三状态,调整原因,...,原始超额净值,调整超额净值,调整相对原始净值,year,month,raw_nav_2026,adjusted_nav_2026,index_nav_2026,adjusted_excess_2026,adjusted_minus_raw_2026
139,2026-08-03,-1,0,1,0,0,0,0,0,-1退出到0,...,2.035521,3.436344,2.400823,2026,2026-08,1.228839,1.361655,1.013107,1.348548,1.132816
140,2026-08-04,0,0,0,0,0,0,0,0,基础0,...,2.022012,3.422834,2.400823,2026,2026-08,1.228839,1.361655,1.026616,1.335039,1.132816
141,2026-08-05,0,0,0,0,0,0,0,0,基础0,...,1.999597,3.400419,2.400823,2026,2026-08,1.228839,1.361655,1.049031,1.312624,1.132816
142,2026-08-06,0,0,0,0,0,0,0,0,基础0,...,1.987810,3.388633,2.400823,2026,2026-08,1.228839,1.361655,1.060818,1.300837,1.132816
143,2026-08-07,0,0,0,0,0,0,0,0,基础0,...,1.965440,3.366263,2.400823,2026,2026-08,1.228839,1.361655,1.083188,1.278467,1.132816
144,2026-08-10,0,0,0,0,0,0,0,0,基础0,...,1.968520,3.369342,2.400823,2026,2026-08,1.228839,1.361655,1.080109,1.281546,1.132816
145,2026-08-11,0,0,0,1,0,0,0,-1,0转-1持有,...,1.968674,3.369652,2.400977,2026,2026-08,1.228839,1.361810,1.079954,1.281856,1.132971
146,2026-08-12,0,0,0,1,0,0,0,-1,0转-1持有,...,1.953782,3.339867,2.386085,2026,2026-08,1.228839,1.346918,1.094846,1.252071,1.118078
147,2026-08-13,0,0,0,1,0,0,0,-1,0转-1持有,...,1.967022,3.366348,2.399325,2026,2026-08,1.228839,1.360158,1.081606,1.278552,1.131319
148,2026-08-14,0,0,0,1,0,0,0,-1,0转-1持有,...,1.964742,3.361786,2.397045,2026,2026-08,1.228839,1.357877,1.083887,1.273990,1.129038


2026 调整后加算收益： 35.47833834278934 %
2026 调整后相对指数超额： 28.875198153705966 %
2026 调整后方向持有日： 82


## 3. 检查 04→05 的口径没有被改变

In [4]:
stage04_detail = pd.read_csv(STAGE04_DIR / 'O2O加算逐日收益与状态.csv', encoding='utf-8-sig', parse_dates=['实际执行日', '推定形成日'])
if not stage04_detail['形成日早于执行日'].all():
    raise AssertionError('04 中存在形成日不早于执行日的行')
if (stage04_detail['O2O可评价'] & stage04_detail['执行日O2O'].isna()).any():
    raise AssertionError('04 将无 O2O 的行错误标记为可评价')
latest = stage04_detail.iloc[-1]
print('最新信号形成日：', latest['推定形成日'])
print('最新信号执行日：', latest['实际执行日'])
print('最新信号是否已有下一开盘价：', bool(latest['O2O可评价']))
print('05 只读取 04 结果，口径检查通过。')

最新信号形成日： 2026-08-24 00:00:00
最新信号执行日： 2026-08-25 00:00:00
最新信号是否已有下一开盘价： False
05 只读取 04 结果，口径检查通过。


输出图中，`15`—`19` 是冻结 1545 内部状态、慢快线背离、年度表现汇总、状态对比和全周期市场环境诊断；若要与云桌面截图完全同截止日，应给 `COMPANY_SPOT_PATH` 指向同一份已更新到最新交易日的本地现货。